# SARBLOH Experimentation: submission run on every game

The agent is `Sarbloh-Experimentation/harness/` (a copy of `Sarbloh/harness/`; upstream kernel `rlm/` unchanged).
Save & Run plays every public game offline; Submit reruns the same notebook on the hidden games through the
competition gateway. Every game is alive from the start and the priority scheduler gives 10 of them the GPU at a
time (SGLang serves 10 requests at once, 139264-token window; the agent compacts above 120k).

The model is Qwen3.8-Flash-Next on SGLang with MTP speculation (`harness/llm/qwen_flash_next.py`); the 27B NVFP4
chain follows as the fallback.

Attach: the competition (ARC wheels + games + the RTX Pro 6000), `banwait13/sarblohagent` (this repo, updated after
the push), the models `dfranzen/intel-qwen3.8-flash-next-w4a16-autoround` (Flash-Next, AutoRound INT4) and
`dfranzen/albucino-qwen3-8-flash-next-drafter` (its MTP drafter), `banwait13/sglangwheels` (SGLang wheels + FR-Spec
map), `banwait13/qwencoder` (nvidia/Qwen3.8-27B-NVFP4, fallback), and for the vLLM FP8 fallback
`fumiyauchiyama/qwen3-8-27b-fp8-hf-snapshot` and `banwait13/sarbloh-vllm-wheelhouse`.
Internet **off**.

## 0. CONFIG: the experiment id and its config diff (overrides of `harness/config.py`)

In [ ]:
import os, time
NOTEBOOK_START = time.time()

EXPERIMENT = "experimentation_submission"   # the ledger label
# Submit to Competition reruns this notebook on the hidden games with KAGGLE_IS_COMPETITION_RERUN set. Its outputs are
# never seen, so tracing is off there. Save & Run (testing) keeps it on: transcripts, pictures, trace.md, reports and
# sft_levels.jsonl for LoRA SFT.
SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
TRACING = not SUBMISSION
# Every game (None = all environments offline; the rerun always plays every hidden game). All games start at once;
# the priority scheduler lets 10 hold the GPU, matching SGLang's --max-running-requests 10, each for 12 LLM calls
# (fewer hand-overs: a game that waits long can lose its cached prompt). Kaggle's 12 h limit is the
# only clock (11.5 h budget, games stop 5 min before it); the move, turn and token limits are far above any run.
EXPERIMENT_OVERRIDES = {
    "games": ["ls20","re86","wa30"], "concurrency": 3, "notebook_budget_s": 41400.0, "game_wall_s": 41400.0,
    "max_actions_per_game": 100000,
    "scheduler": {"enabled": True, "slots": 3, "quantum_calls": 12, "token_scale": 80000.0},
    "agent": {"limits": {"max_turns": 1000000, "max_output_tokens": 10000000000}},
}

BASE = {
    "model": "qwen_flash_next",       # harness/llm/qwen_flash_next.py ("qwen" = the 27B, harness/llm/qwen.py)
    "experiment": EXPERIMENT,
    "teardown_reserve_s": 300.0,
    "record": True,                   # ARC SDK recording + agent step refs -> prime_run/trace.md (off on the rerun)
    "tracing": TRACING,               # transcripts + pictures + harness/trace.py (reports, sft_levels.jsonl)
    # Flash-Next + MTP + FR-Spec on SGLang first, then the 27B NVFP4 chain; each profile that fails start or a smoke
    # test hands over to the next
    "server": {"profile_chain": ["sglang_flash_frspec_vision", "sglang_flash_mtp_vision", "sglang_flash_vision",
                               "sglang_flash", "sglang_nvfp4_mtp5_vision", "sglang_nvfp4_mtp_vision",
                               "sglang_nvfp4_vision", "sglang_nvfp4", "qwen_fp8_vision", "qwen_fp8"]},
}

def merge(a, b):
    out = dict(a)
    for k, v in b.items():
        out[k] = merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else v
    return out

OVERRIDES = merge(BASE, EXPERIMENT_OVERRIDES)
print(f"submission={SUBMISSION} tracing={OVERRIDES['tracing']}")
AGENT_DATASET = "banwait13/sarblohagent"
AGENT_DIR = "Sarbloh-Experimentation"

## 1. ARC runtime (competition wheels, offline) and the agent source

In [ ]:
import os, subprocess, sys
from pathlib import Path

COMPETITION_DIR = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3")
# arc-agi pins its own Pillow; installed over the one Kaggle already loaded, the two mix ("cannot import name '_Ink'
# from 'PIL._typing'"). So install arc-agi and arcengine without their dependencies (Kaggle has them all) and keep
# Kaggle's Pillow. Only if arc-agi then fails to import, install it with its dependencies.
_pip = [sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--no-warn-conflicts", "--disable-pip-version-check",
        "--find-links", str(COMPETITION_DIR / "arc_agi_3_wheels")]
subprocess.check_call(_pip + ["--no-deps", "arc-agi", "arcengine"], stdout=subprocess.DEVNULL)
_probe = "import arc_agi, arcengine, PIL; from PIL import Image, ImageDraw, ImageFont; print('Pillow', PIL.__version__)"
_check = subprocess.run([sys.executable, "-c", _probe], capture_output=True, text=True)
if _check.returncode != 0:
    print("arc-agi without its dependencies failed, installing them:", _check.stderr[-1500:])
    subprocess.check_call(_pip + ["arc-agi"], stdout=subprocess.DEVNULL)
    _check = subprocess.run([sys.executable, "-c", _probe], capture_output=True, text=True)
print(_check.stdout.strip() or _check.stderr[-1500:])
import importlib
importlib.invalidate_caches()
for _m in [m for m in sys.modules if m == "PIL" or m.startswith("PIL.")]:
    del sys.modules[_m]
from PIL import Image, ImageDraw, ImageFont  # a Pillow mix-up fails here, at once, not at the image smoke test

owner, slug = AGENT_DATASET.split("/")
candidates = [Path("/kaggle/input/datasets") / owner / slug, Path("/kaggle/input") / slug]
AGENT_ROOT = next((p for p in candidates if (p / AGENT_DIR / "harness").exists()), None)
if AGENT_ROOT is None:
    hits = [p.parent.parent for p in Path("/kaggle/input").glob(f"**/{AGENT_DIR}/harness/run.py")]
    AGENT_ROOT = hits[0].parent if hits else None
if AGENT_ROOT is None:
    raise FileNotFoundError(f"{AGENT_DIR} not found: attach {AGENT_DATASET} (updated from the git repo)")
for p in (AGENT_ROOT / AGENT_DIR, AGENT_ROOT):
    sys.path.insert(0, str(p))
print("agent root:", AGENT_ROOT)
print(subprocess.run(["ls", "-la", str(AGENT_ROOT / AGENT_DIR)], capture_output=True, text=True).stdout)

## 2. Run: server up (profile chain + smoke test), play the games, teardown

In [ ]:
from harness.run import main

SUMMARY = main(OVERRIDES, notebook_start=NOTEBOOK_START)

## 3. Summary

In [ ]:
import json
print(json.dumps(SUMMARY, indent=1, default=str))
print(f"notebook wall: {time.time() - NOTEBOOK_START:.0f}s")

## 4. Step trace: per game `steps.md` (turn by turn), `steps.jsonl`, SDK recording, SDK scorecard

In [ ]:
trace_md = next(Path("/kaggle/working").rglob("prime_run/trace.md"), None)
if trace_md is None:
    print("no trace.md (tracing off)" if not OVERRIDES["tracing"] else "no trace.md (run crashed before results.json?)")
else:
    print(trace_md.read_text(encoding="utf-8"))
    run_dir = trace_md.parent
    for f in sorted(run_dir.rglob("*")):
        if f.is_file() and f.suffix in (".md", ".jsonl", ".json"):
            print(f"{f.stat().st_size:>10}  {f.relative_to(run_dir)}")
    frames = [f for f in run_dir.rglob("frames/*.png")]
    print(f"{sum(f.stat().st_size for f in frames):>10}  {len(frames)} pictures in games/*/frames/")